# Project 4 — Inference-Faithful Milestone 2 Sweep

This notebook models deployment-time inference more faithfully:
- weights are prequantized offline,
- only activation quantization remains online for `w4a4_nvfp4_inference`,
- and we compare the real quantized workload against an ideal no-overhead quantized lower bound.

Artifacts are written under `project4_m1/sweeps/<shape>/<config>/...` plus sweep-level summary files.

In [ ]:
from pathlib import Path

import pandas as pd

from project4_m1_sweep import (
    AF_AVAILABLE,
    ARCH_PARAMS,
    CONFIGS,
    OUT_DIR,
    ROOT,
    SHAPES,
    SWEEP_DIR,
    load_breakdown_records,
    make_architecture,
    make_workload,
    plot_breakdowns,
    plot_summary,
    run_case,
    run_sweep,
    sanity_check_workloads,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

print(f"Working directory: {Path.cwd()}")
print(f"Lab root: {ROOT}")
print(f"Output directory: {OUT_DIR}")
print(f"Sweep directory: {SWEEP_DIR}")
print(f"AccelForge available: {AF_AVAILABLE}")
print(f"Architecture params: {ARCH_PARAMS}")

## Stable Helper Interface

The sweep is driven through these helpers:
- `make_workload(shape_name, m, n, k, config_name)`
- `make_architecture(rescale_energy_pj=3.7, rescale_latency=2)`
- `run_case(shape_name, dims, config_name)`
- `run_sweep(shapes, configs)`
- `plot_summary(results_df)`
- `plot_breakdowns(breakdown_records)`

In [ ]:
sanity = sanity_check_workloads()
print("Sanity checks passed.")
print("Inference einsums:", sanity["inference_einsums"])
print("Ideal einsums:", sanity["ideal_einsums"])
display(pd.DataFrame.from_dict(SHAPES, orient="index"))
display(pd.DataFrame({"config": CONFIGS}))

## Run The 16-Case Sweep

This executes all 4 shapes × 4 configs, writes per-case workload / arch / mapping artifacts, and emits:
- `project4_m1/sweeps/results_summary.csv`
- `project4_m1/sweeps/results_breakdowns.json`

In [ ]:
results_df = run_sweep(SHAPES, CONFIGS)
print(f"Wrote summary CSV: {SWEEP_DIR / 'results_summary.csv'}")
print(f"Wrote breakdown JSON: {SWEEP_DIR / 'results_breakdowns.json'}")
display(results_df)

## Tables For Proposal Use

These are the three main summary views:
- absolute energy and latency,
- normalized energy and latency versus FP16 baseline,
- and the gap between the real W4A4 inference workload and the ideal no-overhead quantized lower bound.

In [ ]:
absolute_table = results_df[[
    "shape",
    "config",
    "energy_pj",
    "latency_cycles",
    "dominant_energy_einsum",
    "dominant_latency_einsum",
    "status",
]].sort_values(["shape", "config"])

normalized_table = results_df[[
    "shape",
    "config",
    "energy_norm_vs_fp16",
    "latency_norm_vs_fp16",
    "energy_norm_vs_ideal",
    "latency_norm_vs_ideal",
]].sort_values(["shape", "config"])

overhead_gap_table = results_df.loc[
    results_df["config"] == "w4a4_nvfp4_inference",
    [
        "shape",
        "energy_pj",
        "ideal_energy_pj",
        "energy_over_ideal_pj",
        "latency_cycles",
        "ideal_latency_cycles",
        "latency_over_ideal_cycles",
        "dominant_energy_einsum",
        "dominant_latency_einsum",
    ],
].sort_values("shape")

display(absolute_table)
display(normalized_table)
display(overhead_gap_table)

## Plots

`plot_summary(results_df)` renders:
1. grouped bars for normalized energy and latency,
2. a stacked W4A4 inference energy breakdown by shape,
3. and the overhead-to-ideal chart.

`plot_breakdowns(breakdown_records)` renders a per-einsum stacked breakdown for the inference workload.

In [ ]:
figures = plot_summary(results_df)
breakdown_records = load_breakdown_records()
breakdown_fig = plot_breakdowns(breakdown_records)
print("Generated figure keys:", list(figures.keys()))

## Post-Run Checks

These assertions encode the milestone sanity checks directly in the notebook.

In [ ]:
assert len(results_df) == 16, f"Expected 16 sweep rows, found {len(results_df)}"
assert (SWEEP_DIR / "results_summary.csv").exists()
assert (SWEEP_DIR / "results_breakdowns.json").exists()

if results_df["latency_cycles"].notna().all():
    merged = results_df.pivot(index="shape", columns="config", values="latency_cycles")
    assert (merged["w4a4_ideal_no_overhead"] <= merged["w4a4_nvfp4_inference"]).all(), "Ideal latency should be <= inference latency"

if results_df["energy_pj"].notna().all():
    larger_k_shapes = [shape for shape, dims in SHAPES.items() if dims["k"] >= 4096]
    larger_k_df = results_df[(results_df["config"] == "w4a4_nvfp4_inference") & (results_df["shape"].isin(larger_k_shapes))]
    assert not larger_k_df.empty
    assert larger_k_df["dominant_energy_einsum"].fillna("").str.startswith("Rescale").any(), "Expected a rescale stage to dominate energy for at least one larger-k inference case"

print("Post-run checks passed.")